In [ ]:
import sys
import os
from collections import defaultdict
import pandas as pd
import scanpy as sc
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
from glmpca import glmpca
from itertools import combinations
import torch

import sys
from importlib import reload

import gaston
from gaston import neural_net,cluster_plotting, dp_related, segmented_fit, restrict_spots, model_selection
from gaston import binning_and_plotting, isodepth_scaling, run_slurm_scripts, parse_adata
from gaston import spatial_gene_classification, plot_cell_types, filter_genes, process_NN_output

import seaborn as sns
import math
from numpy.random import default_rng

In [ ]:
# Load gene labels
gene_labels = pd.read_csv("../../data/mucosa_retained_rownames.csv")
gene_labels = gene_labels['x'].values
np.save("../data/gene_labels.npy", gene_labels)

In [ ]:
# gaston_model, A, S= process_NN_output.process_files('colorectal_tumor_tutorial_outputs') # model trained above
gaston_model, A, S= process_NN_output.process_files('colorectal_tumor_tutorial_outputs') # MATCH PAPER FIGURES

# May need to re-load counts_mat, coords_mat, and gene_labels
counts_mat=np.load('../data/counts_mat.npy',allow_pickle=True) # download from google drive folder
coords_mat=np.load('../data/coords_mat.npy',allow_pickle=True)
gene_labels=np.load('../data/gene_labels.npy',allow_pickle=True)

In [ ]:
model_selection.plot_ll_curve(gaston_model, A, S, max_domain_num=8, start_from=2,num_buckets=100)

In [ ]:
num_layers=4 # From the above plot
gaston_isodepth, gaston_labels=dp_related.get_isodepth_labels(gaston_model,A,S,num_layers)

# DATASET-SPECIFIC: so domains are ordered with tumor being last
gaston_isodepth= np.max(gaston_isodepth) -1 * gaston_isodepth
gaston_labels=(num_layers-1)-gaston_labels

In [ ]:
show_streamlines=True
rotate = np.radians(-90) # rotate coordinates by -90
arrowsize=2

cluster_plotting.plot_isodepth(gaston_isodepth, S, gaston_model, figsize=(7,6), streamlines=show_streamlines, 
                               rotate=rotate,arrowsize=arrowsize, 
                               neg_gradient=True) # since we did isodepth -> -1*isodepth above, we also need to do gradient -> -1*gradient

In [ ]:
reload(segmented_fit)
# compute piecewise linear fit for restricted spots
pw_fit_dict=segmented_fit.pw_linear_fit(counts_mat.T, gaston_labels, gaston_isodepth,
                                        None, [],  umi_threshold=10, isodepth_mult_factor=0.01,)
# for plotting
binning_output=binning_and_plotting.bin_data(counts_mat.T, gaston_labels, gaston_isodepth, 
                         None, gene_labels, num_bins=15, umi_threshold=10)

In [ ]:
slope_mat, intercept_mat, discont_mat, pv_mat = pw_fit_dict["all_cell_types"]

# These are the genes retained by GASTON after UMI filtering.
gene_labels_idx = binning_output["gene_labels_idx"]

abs_slope_mat = np.abs(slope_mat)

# For each gene, find the layer/domain where its absolute ISODEPTH slope is largest.
best_layer = np.argmax(abs_slope_mat, axis=1)

gaston_gene_rankings = pd.DataFrame({
    "gene": gene_labels_idx,
    "best_layer": best_layer,
    "slope": slope_mat[np.arange(slope_mat.shape[0]), best_layer],
    "abs_slope": abs_slope_mat[np.arange(slope_mat.shape[0]), best_layer],
    "p_value": pv_mat[np.arange(pv_mat.shape[0]), best_layer],
    "min_p_value": np.min(pv_mat, axis=1),
})

gaston_gene_rankings = (
    gaston_gene_rankings
    .sort_values("min_p_value", ascending=True)
    .reset_index(drop=True)
)

gaston_gene_rankings["isodepth_rank"] = np.arange(1, len(gaston_gene_rankings) + 1)

display(gaston_gene_rankings.head(50))

#Save gaston_gene_rankings to data
gaston_gene_rankings.to_csv("../data/gaston_gene_rankings.csv", index=False)

In [ ]:
counts_mat.shape

In [ ]:
pw_fit_dict

In [ ]:
gene_name = "Ddx58"
binning_and_plotting.plot_gene_raw(gene_name, gene_labels, counts_mat, S, vmin=8.5, figsize=(6,4))
plt.title(f'{gene_name} Raw Expression')
binning_and_plotting.plot_gene_function(gene_name, S, pw_fit_dict, gaston_labels, gaston_isodepth, binning_output, figsize=(6,4))
plt.title(f'{gene_name} GASTON Function')
plt.show()

In [ ]:
offset=10**6
binning_and_plotting.plot_gene_pwlinear('Ephb4', pw_fit_dict, gaston_labels, gaston_isodepth, 
                                        binning_output, cell_type_list=None, pt_size=70, 
                                        linear_fit=True, ticksize=15, figsize=(7,3), offset=offset, lw=2,
                                       domain_boundary_plotting=True)

In [ ]:
#Row names of count_mat
gene_labels[300:500]